In [1]:
%pip install pyarrow pandas numpy azure-identity azure-storage-file-datalake -q

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os, subprocess

repo_path = "/home/azureuser/cloudfiles/code/Users/pliao/retail-credit-azure"
if not os.path.exists(repo_path):
    os.system("git clone git@github.com:ping-liao/retail-credit-azure.git " + repo_path)
else:
    result = subprocess.run(["git", "pull"], cwd=repo_path, capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)

Already up to date.




In [3]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "src/prep.py"],
    cwd=repo_path,
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr)

Reading lending-club/accepted_2007_to_2018Q4.csv from bronze...
  Loaded 2,260,701 rows, 27 columns
Cleaning...
  After filtering to closed loans: 1,345,310 rows
  Dropped 1,230 rows with missing critical fields
Engineering features...
Writing lending-club/accepted_cleaned.parquet to silver...
  Done — 1,344,080 rows written to silver/lending-club/accepted_cleaned.parquet

Prep complete.
          loan_amnt      int_rate           dti      fico_mid       default
count  1.344080e+06  1.344080e+06  1.344080e+06  1.344080e+06  1.344080e+06
mean   1.442078e+04  1.323798e+01  1.828495e+01  6.981743e+02  1.996220e-01
std    8.715239e+03  4.768086e+00  1.116122e+01  3.184574e+01  3.997164e-01
min    5.000000e+02  5.310000e+00 -1.000000e+00  6.270000e+02  0.000000e+00
25%    8.000000e+03  9.750000e+00  1.180000e+01  6.720000e+02  0.000000e+00
50%    1.200000e+04  1.274000e+01  1.762000e+01  6.920000e+02  0.000000e+00
75%    2.000000e+04  1.599000e+01  2.406000e+01  7.120000e+02  0.000000e+00
m

In [4]:
from azure.storage.filedatalake import DataLakeServiceClient
from azure.identity import DefaultAzureCredential

service_client = DataLakeServiceClient(
    account_url="https://stretailcreditrc01.dfs.core.windows.net",
    credential=DefaultAzureCredential()
)
paths = service_client.get_file_system_client("silver").get_paths("lending-club")
for p in paths:
    print(p.name, f"  {round(p.content_length/1024/1024, 1)} MB")

lending-club/accepted_cleaned.parquet   38.9 MB
